<a href="https://colab.research.google.com/github/etcex2969-spec/-AIFFEL_quest_eng/blob/main/PIPLINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install structlog -q
import os
from rag_pipeline import RAGConfig, create_pipeline

# 설정값 변경: API 주소를 더미로 바꾸거나 로컬 설정을 주입
custom_config = RAGConfig(
    rag_api_base="http://localhost",  # 실제 호출을 피하기 위한 설정
    rag_context_snippet="[실제 데이터] nbconvert 타임아웃은 대용량 노트북 처리 시 리소스 부족으로 발생하며, --ExecutePreprocessor.timeout 설정을 통해 해결할 수 있습니다."
)

# 파이프라인 생성 및 실행
try:
    pipeline = create_pipeline(config=custom_config)
    user_q = "주말 깃허브 서버의 nbconvert 타임아웃 문제 원인과 해결책은?"

    # RAGClient의 동작을 모킹(Mocking)하여 외부 통신 없이 진행
    from unittest.mock import patch
    with patch('rag_pipeline.RAGClient.retrieve') as mock_retrieve:
        mock_retrieve.return_value = custom_config.rag_context_snippet
        output = pipeline.run(user_q)
        print("\n[실행 결과]")
        print(output)
except Exception as e:
    print(f"실행 중 오류 발생: {e}")

{"component": "RAGPipeline", "question_preview": "\uc8fc\ub9d0 \uae43\ud5c8\ube0c \uc11c\ubc84\uc758 nbconvert \ud0c0\uc784\uc544\uc6c3 \ubb38\uc81c \uc6d0\uc778\uacfc \ud574\uacb0\ucc45\uc740?", "event": "pipeline_start", "request_id": "4b0fa1d2-6804-4ac4-bba9-0c401edef07a", "level": "info", "timestamp": "2026-06-03T10:20:10.052539Z"}
{"component": "RAGPipeline", "event": "step_1_init_agent", "request_id": "4b0fa1d2-6804-4ac4-bba9-0c401edef07a", "level": "info", "timestamp": "2026-06-03T10:20:10.053253Z"}
{"component": "RAGPipeline", "tool": "retrieval_tool", "event": "step_2_decide_tool", "request_id": "4b0fa1d2-6804-4ac4-bba9-0c401edef07a", "level": "info", "timestamp": "2026-06-03T10:20:10.053869Z"}
{"component": "RAGPipeline", "event": "step_3_run_rag", "request_id": "4b0fa1d2-6804-4ac4-bba9-0c401edef07a", "level": "info", "timestamp": "2026-06-03T10:20:10.054475Z"}
{"prompt_length": 596, "event": "prompt_built", "request_id": "4b0fa1d2-6804-4ac4-bba9-0c401edef07a", "level": "info

### 1. API 키 설정 방법

두 가지 방법 중 편한 방법을 선택하세요:
- **방법 A (추천):** 왼쪽 사이드바의 🔑 아이콘(Secrets)을 클릭하고 `OPENAI_API_KEY`라는 이름으로 키를 추가한 뒤 'Notebook access'를 켭니다.
- **방법 B:** 아래 셀에서 직접 환경 변수를 설정합니다.

In [23]:
import os
from google.colab import userdata

try:
    # Get the key and strip any hidden whitespace or newlines
    raw_key = userdata.get('OPENAI_API_KEY')
    if raw_key:
        os.environ["OPENAI_API_KEY"] = raw_key.strip()
        print("✅ Secrets에서 API 키를 성공적으로 로드하고 공백을 제거했습니다.")
    else:
        print("⚠️ Secrets에서 'OPENAI_API_KEY'를 찾을 수 없습니다.")
except Exception as e:
    print(f"⚠️ 오류 발생: {e}")

✅ Secrets에서 API 키를 성공적으로 로드하고 공백을 제거했습니다.


### 2. 파이프라인 재실행

이제 설정된 키를 사용하여 `RAGConfig`를 생성하고 실제 LLM 호출을 진행합니다.

In [24]:
from rag_pipeline import RAGConfig, create_pipeline
from unittest.mock import patch
import openai
import os

# 1. API 키 확인 (환경 변수에서 최신 상태 로드)
api_key = os.environ.get("OPENAI_API_KEY", "")

if not api_key or len(api_key) < 20:
    print("❌ 유효한 OPENAI_API_KEY가 설정되지 않았습니다. Secrets 설정을 확인해주세요.")
else:
    # 2. 이미 strip() 처리된 키를 사용하여 설정 생성
    config = RAGConfig(
        openai_api_key=api_key,
        rag_context_snippet="[실제 데이터] nbconvert 타임아웃은 대용량 노트북 처리 시 리소스 부족으로 발생하며, --ExecutePreprocessor.timeout 설정을 통해 해결할 수 있습니다."
    )

    # 3. 파이프라인 생성
    pipeline = create_pipeline(config=config)

    # 4. RAG 검색 부분만 모킹하여 실행
    with patch('rag_pipeline.RAGClient.retrieve') as mock_retrieve:
        mock_retrieve.return_value = config.rag_context_snippet

        print(f"🚀 실제 LLM 호출을 시작합니다... (키 길이: {len(api_key)})")
        try:
            output = pipeline.run("주말 깃허브 서버의 nbconvert 타임아웃 문제 원인과 해결책은?")
            print("\n[최종 답변]")
            print(output)
        except openai.AuthenticationError:
            print("\n❌ 인증 오류: API 키가 여전히 거부되었습니다. 키 자체가 만료되었거나 잘못되었을 수 있습니다.")
        except Exception as e:
            print(f"\n❌ 오류 발생: {e}")

🚀 실제 LLM 호출을 시작합니다... (키 길이: 164)
{"component": "RAGPipeline", "question_preview": "\uc8fc\ub9d0 \uae43\ud5c8\ube0c \uc11c\ubc84\uc758 nbconvert \ud0c0\uc784\uc544\uc6c3 \ubb38\uc81c \uc6d0\uc778\uacfc \ud574\uacb0\ucc45\uc740?", "event": "pipeline_start", "request_id": "3f3e46d1-f8e5-4e8e-b474-64f0c1ef0fcb", "level": "info", "timestamp": "2026-06-03T10:20:10.621993Z"}
{"component": "RAGPipeline", "event": "step_1_init_agent", "request_id": "3f3e46d1-f8e5-4e8e-b474-64f0c1ef0fcb", "level": "info", "timestamp": "2026-06-03T10:20:10.624120Z"}
{"component": "RAGPipeline", "tool": "retrieval_tool", "event": "step_2_decide_tool", "request_id": "3f3e46d1-f8e5-4e8e-b474-64f0c1ef0fcb", "level": "info", "timestamp": "2026-06-03T10:20:10.624920Z"}
{"component": "RAGPipeline", "event": "step_3_run_rag", "request_id": "3f3e46d1-f8e5-4e8e-b474-64f0c1ef0fcb", "level": "info", "timestamp": "2026-06-03T10:20:10.626040Z"}
{"prompt_length": 596, "event": "prompt_built", "request_id": "3f3e46d1-f8e5-4e8e-

In [25]:
import pandas as pd

# 1. 추천 데이터셋(FAQ) 생성
data = {
    "question": [
        "nbconvert 타임아웃 문제",
        "CUDA Out of Memory 오류",
        "GitHub Actions 권한 설정",
        "Docker 빌드 속도 최적화"
    ],
    "answer": [
        "nbconvert 타임아웃은 대용량 파일 처리 시 발생하며, --ExecutePreprocessor.timeout=600 옵션을 추가하여 해결합니다.",
        "GPU 메모리 부족은 batch_size를 줄이거나 torch.cuda.empty_cache()를 호출하여 완화할 수 있습니다.",
        "GitHub Actions에서 GITHUB_TOKEN 권한은 workflow 파일의 permissions 섹션에서 설정합니다.",
        "Docker 레이어 캐싱을 활용하고, 멀티 스테이지 빌드를 사용하면 이미지 크기를 줄이고 속도를 높일 수 있습니다."
    ]
}

df_faq = pd.DataFrame(data)
df_faq.to_csv('faq_dataset.csv', index=False)
print("✅ 'faq_dataset.csv' 데이터셋이 생성되었습니다.")
display(df_faq)

✅ 'faq_dataset.csv' 데이터셋이 생성되었습니다.


,question,answer
0,nbconvert 타임아웃 문제,"nbconvert 타임아웃은 대용량 파일 처리 시 발생하며, --ExecutePre..."
1,CUDA Out of Memory 오류,GPU 메모리 부족은 batch_size를 줄이거나 torch.cuda.empty_...
2,GitHub Actions 권한 설정,GitHub Actions에서 GITHUB_TOKEN 권한은 workflow 파일의...
3,Docker 빌드 속도 최적화,"Docker 레이어 캐싱을 활용하고, 멀티 스테이지 빌드를 사용하면 이미지 크기를 ..."


In [26]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_faq)

https://docs.google.com/spreadsheets/d/1GwwibWYk8cwOo907F2ql6pArMpAAuu4AvI_kW2uTRfk/edit#gid=0


In [27]:
# google.colab.sheets 인증 오류 시 대안으로 사용
try:
    from google.colab import sheets
    sheet = sheets.InteractiveSheet(df=df_faq)
except Exception:
    print("💡 구글 시트 인증에 실패하여 기본 데이터 프레임 출력으로 대체합니다.")
    display(df_faq)

https://docs.google.com/spreadsheets/d/1-9AO7GDtaoYeaPzPiJkDn2keg6uPebVXs-zdfcKoJi4/edit#gid=0


In [28]:
import pandas as pd
from rag_pipeline import RAGConfig, create_pipeline
from unittest.mock import patch
import os

# 2. 데이터셋 로드 및 검색 시뮬레이션
df = pd.read_csv('faq_dataset.csv')

def simple_retriever(query):
    # 질문 키워드가 포함된 가장 적절한 답변을 찾는 간단한 로직
    for i, row in df.iterrows():
        if any(word in query for word in row['question'].split()):
            return f"[검색된 지식] {row['answer']}"
    return "[검색된 지식] 관련 정보를 찾을 수 없습니다."

# 3. 파이프라인 실행
api_key = os.environ.get("OPENAI_API_KEY", "")
query = "주말에 nbconvert 타임아웃 에러가 났는데 어떻게 하죠?"
retrieved_context = simple_retriever(query)

config = RAGConfig(
    openai_api_key=api_key,
    rag_context_snippet=retrieved_context
)

pipeline = create_pipeline(config=config)

with patch('rag_pipeline.RAGClient.retrieve') as mock_retrieve:
    mock_retrieve.return_value = config.rag_context_snippet

    print(f"🔍 검색된 컨텍스트: {retrieved_context}")
    print("🚀 LLM 답변 생성 중...")
    try:
        output = pipeline.run(query)
        print("\n[최종 답변]")
        print(output)
    except Exception as e:
        print(f"\n❌ 실행 중 오류: {e}")

🔍 검색된 컨텍스트: [검색된 지식] nbconvert 타임아웃은 대용량 파일 처리 시 발생하며, --ExecutePreprocessor.timeout=600 옵션을 추가하여 해결합니다.
🚀 LLM 답변 생성 중...
{"component": "RAGPipeline", "question_preview": "\uc8fc\ub9d0\uc5d0 nbconvert \ud0c0\uc784\uc544\uc6c3 \uc5d0\ub7ec\uac00 \ub0ac\ub294\ub370 \uc5b4\ub5bb\uac8c \ud558\uc8e0?", "event": "pipeline_start", "request_id": "756960c7-4427-4f12-8cda-d1e8fc5b55ec", "level": "info", "timestamp": "2026-06-03T10:20:38.259685Z"}
{"component": "RAGPipeline", "event": "step_1_init_agent", "request_id": "756960c7-4427-4f12-8cda-d1e8fc5b55ec", "level": "info", "timestamp": "2026-06-03T10:20:38.260305Z"}
{"component": "RAGPipeline", "tool": "retrieval_tool", "event": "step_2_decide_tool", "request_id": "756960c7-4427-4f12-8cda-d1e8fc5b55ec", "level": "info", "timestamp": "2026-06-03T10:20:38.261968Z"}
{"component": "RAGPipeline", "event": "step_3_run_rag", "request_id": "756960c7-4427-4f12-8cda-d1e8fc5b55ec", "level": "info", "timestamp": "2026-06-03T10:20:38.262452Z"}
{"prompt_leng

In [29]:
import os

def mask_key(key_str):
    if not key_str:
        return 'None'
    if len(key_str) <= 10:
        return '*** (Too short)'
    return f"{key_str[:7]}...{key_str[-4:]}"

print("--- 현재 환경 변수(os.environ) 상태 확인 ---")
api_key = os.environ.get('OPENAI_API_KEY', '')

print(f"OPENAI_API_KEY: {mask_key(api_key)}")
print(f"키 길이: {len(api_key)}")

# 제어 문자나 공백이 포함되어 있는지 최종 체크
if not api_key:
    print("❌ 키가 설정되지 않았습니다.")
elif api_key != api_key.strip():
    print("⚠️ 경고: 여전히 공백이나 줄바꿈 문자가 포함되어 있습니다. (재처리가 필요합니다)")
else:
    print("✅ 확인 완료: 키가 깨끗하게 로드되었습니다.")

--- 현재 환경 변수(os.environ) 상태 확인 ---
OPENAI_API_KEY: sk-proj...LX4A
키 길이: 164
✅ 확인 완료: 키가 깨끗하게 로드되었습니다.


### 💡 API 키 오류 디버깅 가이드

`401 - Incorrect API key provided` 오류는 주로 다음 원인으로 발생합니다. 아래 단계를 따라 문제를 해결해 보세요.

1.  **Colab Secrets 확인 (가장 중요)**:
    *   왼쪽 사이드바의 🔑 아이콘을 클릭하여 Secrets 탭을 엽니다.
    *   `OPENAI_API_KEY` 변수가 정확히 등록되어 있는지 확인합니다.
    *   키 값 자체에 **불필요한 공백이나 줄바꿈 문자**가 포함되어 있지 않은지 다시 한번 확인합니다. (복사/붙여넣기 시 자주 발생)
    *   해당 키의 'Notebook access'가 켜져 있는지 확인합니다.

2.  **`os.environ` 값 재확인**:
    *   셀 `1051c440`의 출력 `OPENAI_API_KEY`와 `키 길이`를 다시 확인하세요.
    *   `mask_key` 함수가 마스킹한 부분(예: `sk-proj...LX4A`)을 제외하고도, 키의 시작과 끝 부분이 정확히 로드되었는지 시각적으로 검토해 보세요.
    *   `키 길이`가 OpenAI에서 발급받은 실제 키의 길이와 일치하는지 확인하세요. (일반적으로 `sk-proj-`로 시작하는 키는 약 164~168자입니다.)

3.  **`rag_pipeline.py` 코드 흐름 이해**:
    *   `f837191d` 셀의 `RAGConfig` 클래스를 보면 `openai_api_key` 필드가 `OPENAI_API_KEY` 환경 변수에서 값을 가져오도록 설정되어 있습니다.
    *   `create_pipeline` 함수가 이 `RAGConfig`를 통해 `OpenAILLMClient`를 초기화합니다.
    *   즉, `os.environ["OPENAI_API_KEY"]`에 올바른 값이 있어야만 파이프라인이 정상 작동합니다.

4.  **`strip()` 함수 동작 확인**:
    *   `fcd35876` 셀에서 `raw_key.strip()`을 통해 공백을 제거하고 있습니다. 이 코드가 정상적으로 작동했는지 `✅ Secrets에서 API 키를 성공적으로 로드하고 공백을 제거했습니다.` 메시지로 확인하세요.
    *   만약 여전히 문제가 있다면, `os.environ[

In [31]:
import openai
import os

# 현재 설정된 API 키 가져오기
api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    print("❌ 환경 변수에 OPENAI_API_KEY가 설정되어 있지 않습니다.")
else:
    print(f"✅ API 키가 환경 변수에 설정되어 있습니다. (길이: {len(api_key)})")
    try:
        # 간단한 OpenAI API 호출 시도 (모델 목록 조회)
        openai.api_key = api_key
        models = openai.models.list()
        print("✅ OpenAI API에 성공적으로 연결되었습니다!")
        # print("사용 가능한 모델의 일부:")
        # for model in models.data[:5]: # 상위 5개 모델만 출력
        #     print(f"- {model.id}")
    except openai.AuthenticationError:
        print("❌ OpenAI API 인증에 실패했습니다. 키가 유효하지 않거나 만료되었을 수 있습니다.")
    except Exception as e:
        print(f"⚠️ 예상치 못한 오류 발생: {e}")


✅ API 키가 환경 변수에 설정되어 있습니다. (길이: 164)
❌ OpenAI API 인증에 실패했습니다. 키가 유효하지 않거나 만료되었을 수 있습니다.


In [37]:
import uuid
import logging
from typing import Optional, Protocol, runtime_checkable

import requests
import structlog
from pydantic import BaseModel, Field
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
)

# ---------------------------------------------------------------------------
# 1. 설정 (Pydantic) — 환경 변수에서 자동 로드
# ---------------------------------------------------------------------------

class RAGConfig(BaseModel):
    """모든 설정을 한 곳에서 관리. 환경 변수 또는 직접 주입 가능."""

    rag_api_base: str = Field(
        default="https://rag-tool.example.com/api",
        validation_alias="RAG_API_BASE",
    )
    rag_api_key: str = Field(default="", validation_alias="RAG_API_KEY")
    rag_tool_name: str = Field(
        default="retrieval_tool", validation_alias="RAG_TOOL_NAME"
    )
    rag_context_snippet: str = Field(
        default="[RAG 지식] 기본 컨텍스트입니다.",
        validation_alias="RAG_CONTEXT_SNIPPET",
    )
    openai_api_key: str = Field(default="", validation_alias="OPENAI_API_KEY")
    openai_model: str = Field(default="gpt-4o-mini", validation_alias="OPENAI_MODEL")

    # 재시도 설정
    retry_attempts: int = Field(default=3, validation_alias="RAG_RETRY_COUNT")
    retry_min_wait: float = Field(default=1.0, validation_alias="RAG_BACKOFF_SEC")
    retry_max_wait: float = Field(default=10.0, validation_alias="RAG_MAX_BACKOFF_SEC")
    timeout_sec: float = Field(default=5.0, validation_alias="RAG_TIMEOUT_SEC")

    model_config = {"populate_by_name": True}

    @classmethod
    def from_env(cls) -> "RAGConfig":
        """환경 변수에서 설정을 로드합니다."""
        import os
        return cls(
            RAG_API_BASE=os.getenv("RAG_API_BASE", "https://rag-tool.example.com/api"),
            RAG_API_KEY=os.getenv("RAG_API_KEY", ""),
            RAG_TOOL_NAME=os.getenv("RAG_TOOL_NAME", "retrieval_tool"),
            RAG_CONTEXT_SNIPPET=os.getenv("RAG_CONTEXT_SNIPPET", "[RAG 지식] 기본 컨텍스트입니다."),
            OPENAI_API_KEY=os.getenv("OPENAI_API_KEY", ""),
            OPENAI_MODEL=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            RAG_RETRY_COUNT=int(os.getenv("RAG_RETRY_COUNT", "3")),
            RAG_BACKOFF_SEC=float(os.getenv("RAG_BACKOFF_SEC", "1.0")),
            RAG_MAX_BACKOFF_SEC=float(os.getenv("RAG_MAX_BACKOFF_SEC", "10.0")),
            RAG_TIMEOUT_SEC=float(os.getenv("RAG_TIMEOUT_SEC", "5.0")),
        )


# ---------------------------------------------------------------------------
# 2. 로깅 설정 (structlog)
# ---------------------------------------------------------------------------

def configure_logging():
    """structlog 로깅을 구성합니다."""
    structlog.configure(
        processors=[
            structlog.stdlib.add_logger_name,
            structlog.stdlib.add_log_level,
            structlog.processors.TimeStamper(fmt="iso"),
            structlog.processors.StackInfoRenderer(),
            structlog.dev.ConsoleRenderer() if os.isatty(1) else structlog.processors.JSONRenderer(),
        ],
        wrapper_class=structlog.stdlib.BoundLogger,
        logger_factory=structlog.stdlib.LoggerFactory(),
        cache_logger_on_first_use=True,
    )
    # 표준 로깅 라이브러리의 핸들러를 설정하여 structlog와 통합합니다.
    # 이를 통해 Tenacity와 같은 라이브러리의 로그도 structlog 포맷으로 출력됩니다.
    if logging.root.handlers:
        for handler in logging.root.handlers:
            logging.root.removeHandler(handler)
    logging.basicConfig(handlers=[logging.StreamHandler()], level=logging.INFO)

configure_logging()


# ---------------------------------------------------------------------------
# 3. RAG 클라이언트 (검색 도구)
# ---------------------------------------------------------------------------

class RAGClient:
    """외부 RAG API와 통신하는 클라이언트."""

    def __init__(self, config: RAGConfig):
        self._config = config
        self._log = structlog.get_logger(__name__)
        self._session = requests.Session()
        if self._config.rag_api_key:
            self._session.headers.update({"X-API-KEY": self._config.rag_api_key})

    @retry(
        retry=retry_if_exception_type(requests.exceptions.RequestException),
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=1, max=10),
        before_sleep=before_sleep_log(structlog.get_logger(__name__), logging.INFO),
        reraise=True,
    )
    def retrieve(self, query: str) -> str:
        """RAG API에서 관련 컨텍스트를 검색합니다."""
        self._log.info("rag_retrieve_start", query=query[:80])
        try:
            response = self._session.post(
                f"{self._config.rag_api_base}/retrieve",
                json={"query": query},
                timeout=self._config.timeout_sec,
            )
            response.raise_for_status()  # HTTP 오류 발생 시 예외 발생
            data = response.json()
            context = data.get("context", "")
            self._log.info("rag_retrieve_success", context_len=len(context))
            return context
        except requests.exceptions.RequestException as e:
            self._log.error("rag_retrieve_failed", error=str(e), status_code=getattr(e.response, 'status_code', 'N/A'))
            raise


# ---------------------------------------------------------------------------
# 4. LLM 클라이언트 (추론 엔진)
# ---------------------------------------------------------------------------

@runtime_checkable
class LLMClient(Protocol):
    """LLM 클라이언트를 위한 프로토콜."""

    def infer(self, prompt: str, temperature: float) -> str:
        ... # pragma: no cover


class OpenAILLMClient(LLMClient):
    """OpenAI API를 사용하여 추론을 수행하는 클라이언트."""

    def __init__(self, config: RAGConfig):
        import openai
        self._config = config
        self._log = structlog.get_logger(__name__)
        # 키가 없으면 시뮬레이션 모드로 전환 (오류 방지)
        if not self._config.openai_api_key:
            self._log.warning("openai_key_missing", fallback="simulation_mode")
            self._simulation_mode = True
        else:
            self._simulation_mode = False
            openai.api_key = self._config.openai_api_key
            # Correctly set openai.base_url to the official OpenAI API endpoint
            # Ensure a trailing slash for correct path construction by the OpenAI library.
            openai.base_url = "https://api.openai.com/v1/"

    @retry(
        retry=retry_if_exception_type(openai.APITimeoutError) | retry_if_exception_type(requests.exceptions.RequestException),
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=1, max=10),
        before_sleep=before_sleep_log(structlog.get_logger(__name__), logging.INFO),
        reraise=True,
    )
    def infer(self, prompt: str, temperature: float) -> str:
        if self._simulation_mode:
            self._log.warning("llm_infer_simulation", prompt_preview=prompt[:80])
            return f"[시뮬레이션] 프롬프트 수신 완료 (temperature={temperature})"

        # Mask the API key for logging purposes
        masked_api_key = f"{self._config.openai_api_key[:7]}...{self._config.openai_api_key[-4:]}" if self._config.openai_api_key else "None"
        self._log.info("llm_infer_start", model=self._config.openai_model, masked_api_key=masked_api_key)

        try:
            response = openai.chat.completions.create(
                model=self._config.openai_model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            self._log.info("llm_infer_success", model=self._config.openai_model, usage=response.usage.model_dump())
            return response.choices[0].message.content
        except openai.AuthenticationError as e:
            self._log.exception("llm_infer_failed_auth", error=str(e), masked_api_key=masked_api_key)
            raise
        except openai.APIStatusError as e:
            self._log.error("llm_infer_failed_api_status", error=f"Error code: {e.status_code} - {e.response.json() if e.response else 'No response'}", model=self._config.openai_model)
            raise
        except Exception as e:
            self._log.exception("llm_infer_failed_general", error=str(e), model=self._config.openai_model)
            raise


# ---------------------------------------------------------------------------
# 5. 프롬프트 빌더
# ---------------------------------------------------------------------------

FEW_SHOT_EXAMPLES = """
[예시 1]
질문: GitHub Actions에서 Python 버전을 지정하는 방법은?
답변: `actions/setup-python` 액션에서 `python-version: '3.11'`로 지정합니다.

[예시 2]
질문: 존재하지 않는 기능에 대한 질문입니다.
답변: 죄송합니다. 제공된 컨텍스트에서 해당 정보를 찾을 수 없습니다.
"""

SYSTEM_PROMPT_TEMPLATE = """당신은 소프트웨어 엔지니어링 및 MLOps 분야의 전문 어시스턴트입니다.
아래 규칙을 반드시 따르세요:
1. 오직 [참고 컨텍스트]에 있는 정보만을 근거로 답변하세요.
2. 컨텍스트에 답이 없으면 "제공된 컨텍스트에서 해당 정보를 찾을 수 없습니다."라고 답하세요.
3. 답변은 간결하고 명확하게, 코드가 필요하면 코드 블록(```)으로 감싸세요.
4. 불확실한 내용을 추측하거나 지어내지 마세요 (No Hallucination).

{few_shot_examples}

[참고 컨텍스트]
{context}

[질문]
{question}

[답변]"""

def build_prompt(user_question: str, context: str) -> str:
    """주어진 질문과 컨텍스트로 LLM 프롬프트를 구성합니다."""
    prompt = SYSTEM_PROMPT_TEMPLATE.format(
        few_shot_examples=FEW_SHOT_EXAMPLES,
        context=context,
        question=user_question,
    )
    return prompt


# ---------------------------------------------------------------------------
# 6. RAG 파이프라인 (오케스트레이션)
# ---------------------------------------------------------------------------

class RAGPipeline:
    """RAG 시스템의 전체 흐름을 관리하는 클래스."""

    def __init__(self, config: RAGConfig, rag_client: RAGClient, llm_client: LLMClient):
        self._config = config
        self._rag_client = rag_client
        self._llm_client = llm_client
        self._log = structlog.get_logger(__name__)

    def init_agent(self) -> str:
        """RAG 에이전트를 초기화하고 기본 컨텍스트를 반환합니다."""
        self._log.info("step_1_init_agent")
        return self._config.rag_context_snippet

    def decide_tool(self) -> str:
        """사용할 도구를 결정합니다. 여기서는 항상 검색 도구를 사용합니다."""
        self._log.info("step_2_decide_tool", tool=self._config.rag_tool_name)
        return self._config.rag_tool_name

    def run_rag(self, initial_context: str, user_question: str) -> str:
        """RAG 검색을 수행하고 컨텍스트를 업데이트합니다."""
        self._log.info("step_3_run_rag")
        retrieved_context = self._rag_client.retrieve(user_question)
        # 기존 컨텍스트와 검색된 컨텍스트를 조합 (여기서는 간단히 덮어씀)
        return retrieved_context if retrieved_context else initial_context

    def model_infer(self, prompt: str, temperature: float = 0.1) -> str:
        self._log.info("step_5_model_infer", temperature=temperature)
        return self._llm_client.infer(prompt, temperature)

    # --- 전체 파이프라인 실행 ---

    def run(self, user_question: str) -> str:
        """
        request_id를 생성해 로그 전 구간에 바인딩한 뒤 파이프라인을 실행합니다.
        init → decide → run_rag → build_prompt → model_infer
        """
        request_id = str(uuid.uuid4())
        structlog.contextvars.bind_contextvars(request_id=request_id)

        self._log.info("pipeline_start", question_preview=user_question[:80])

        try:
            context = self.init_agent()
            tool = self.decide_tool()

            if tool == self._config.rag_tool_name:
                context = self.run_rag(context, user_question)

            prompt = build_prompt(user_question, context)
            answer = self.model_infer(prompt, temperature=0.1)

            self._log.info("pipeline_complete")
            return answer

        except Exception as exc:
            self._log.error("pipeline_error", error=str(exc))
            raise
        finally:
            structlog.contextvars.unbind_contextvars("request_id")


# ---------------------------------------------------------------------------
# 8. 팩토리 함수 — 기본 설정으로 파이프라인 생성
# ---------------------------------------------------------------------------

def create_pipeline(config: Optional[RAGConfig] = None) -> RAGPipeline:
    """환경 변수 기반으로 파이프라인을 생성하는 편의 함수."""
    if config is None:
        config = RAGConfig.from_env()
    rag_client = RAGClient(config)
    llm_client = OpenAILLMClient(config)
    return RAGPipeline(config, rag_client, llm_client)


# ---------------------------------------------------------------------------
# 9. 엔트리포인트
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # To prevent connection errors to a dummy RAG API, we will mock the RAGClient.retrieve method
    # This allows the pipeline to proceed to LLM inference for API key debugging.
    from unittest.mock import patch, MagicMock
    import os

    # Use an actual API key from environment for LLM part if available
    api_key = os.environ.get("OPENAI_API_KEY", "")

    # Create a config that reflects the mocked RAG and potential real LLM calls
    test_config = RAGConfig(
        openai_api_key=api_key,
        rag_api_base="http://localhost", # This will be ignored by the mock RAGClient
        rag_context_snippet="[테스트 컨텍스트] nbconvert 타임아웃 문제는 리소스 부족으로 발생합니다."
    )

    # Patch the RAGClient class in the current module's namespace
    # This ensures that when create_pipeline is called, it gets our mock RAGClient.
    with patch(f'{__name__}.RAGClient') as MockRAGClientClass:
        mock_rag_client_instance = MagicMock()
        mock_rag_client_instance.retrieve.return_value = test_config.rag_context_snippet
        MockRAGClientClass.return_value = mock_rag_client_instance

        pipeline = create_pipeline(config=test_config)
        user_q = "주말 깃허브 서버의 nbconvert 타임아웃 문제 원인과 해결책은?"

        try:
            output = pipeline.run(user_q)
            print(output)
        except Exception as e:
            print(f"실행 중 오류 발생: {e}")


INFO:__main__:{"question_preview": "\uc8fc\ub9d0 \uae43\ud5c8\ube0c \uc11c\ubc84\uc758 nbconvert \ud0c0\uc784\uc544\uc6c3 \ubb38\uc81c \uc6d0\uc778\uacfc \ud574\uacb0\ucc45\uc740?", "event": "pipeline_start", "logger": "__main__", "level": "info", "timestamp": "2026-06-03T10:30:52.256369Z"}
INFO:__main__:{"event": "step_1_init_agent", "logger": "__main__", "level": "info", "timestamp": "2026-06-03T10:30:52.257776Z"}
INFO:__main__:{"tool": "retrieval_tool", "event": "step_2_decide_tool", "logger": "__main__", "level": "info", "timestamp": "2026-06-03T10:30:52.258351Z"}
INFO:__main__:{"event": "step_3_run_rag", "logger": "__main__", "level": "info", "timestamp": "2026-06-03T10:30:52.259315Z"}
INFO:__main__:{"temperature": 0.1, "event": "step_5_model_infer", "logger": "__main__", "level": "info", "timestamp": "2026-06-03T10:30:52.259920Z"}
INFO:__main__:{"model": "gpt-4o-mini", "masked_api_key": "sk-proj...LX4A", "event": "llm_infer_start", "logger": "__main__", "level": "info", "timestam

실행 중 오류 발생: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************LX4A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
